# Hub Identification

In [1]:
import sys

import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"   # GPU slot -- adjust per machine; independent of which model is active below
os.environ["DGLBACKEND"] = "pytorch"

import torch

DEVICE = torch.device("cuda:0")

print(torch.cuda.get_device_name(0))
print(torch.cuda.get_device_capability(0))

NVIDIA RTX PRO 6000 Blackwell Max-Q Workstation Edition
(12, 0)


## 1. Model initialization

In [2]:
# ==================== DeepSeek-R1-Distill-Qwen-14B (ACTIVE) ====================
import torch
from collections import defaultdict
import statistics as stats
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

from dotenv import load_dotenv
from huggingface_hub import login

load_dotenv()
token = os.getenv("HF_TOKEN")
login(token=token)

MODEL_ID = "deepseek-ai/DeepSeek-R1-Distill-Qwen-14B"
MODEL_NAME = "deepseek"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",       # spreads layers across available GPU(s), offloads to CPU if needed
    torch_dtype=torch.bfloat16,
)
model.eval()
print("Model loaded.")


# ==================== Qwen3-14B ====================
# import torch
# from collections import defaultdict
# import statistics as stats
# from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
#
# from dotenv import load_dotenv
# from huggingface_hub import login
#
# load_dotenv()
# token = os.getenv("HF_TOKEN")
# login(token=token)
#
# MODEL_ID = "Qwen/Qwen3-14B"
# MODEL_NAME = "qwen3"
#
# bnb_config = BitsAndBytesConfig(
#     load_in_4bit=True,
#     bnb_4bit_quant_type="nf4",
#     bnb_4bit_compute_dtype=torch.bfloat16,
#     bnb_4bit_use_double_quant=True,
# )
#
# tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
#
# model = AutoModelForCausalLM.from_pretrained(
#     MODEL_ID,
#     quantization_config=bnb_config,
#     device_map="auto",
#     torch_dtype=torch.bfloat16,
# )
# model.eval()
# print("Model loaded.")


# ==================== Qwen2.5-14B-Instruct ====================
# import torch
# from collections import defaultdict
# import statistics as stats
# from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
#
# from dotenv import load_dotenv
# from huggingface_hub import login
#
# load_dotenv()
# token = os.getenv("HF_TOKEN")
# login(token=token)
#
# MODEL_ID = "Qwen/Qwen2.5-14B-Instruct"
# MODEL_NAME = "qwen2.5"
#
# bnb_config = BitsAndBytesConfig(
#     load_in_4bit=True,
#     bnb_4bit_quant_type="nf4",
#     bnb_4bit_compute_dtype=torch.bfloat16,
#     bnb_4bit_use_double_quant=True,
# )
#
# tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
#
# model = AutoModelForCausalLM.from_pretrained(
#     MODEL_ID,
#     quantization_config=bnb_config,
#     device_map="auto",
#     torch_dtype=torch.bfloat16,
# )
# model.eval()
# print("Model loaded.")


# ==================== GPT-Luna (API) ====================
# import os
# from dotenv import load_dotenv
# from openai import OpenAI
#
# load_dotenv()
#
# api_key = os.getenv("GPT_LUNA")
# endpoint = os.getenv("END_POINT")
#
# MODEL_ID = "GPT-5.6-LUNA"
# MODEL_NAME = "gpt_luna"
#
# client = OpenAI(
#     base_url=f"{endpoint}/openai/v1/",
#     api_key=api_key,
# )
#
# response = client.chat.completions.create(
#     model=MODEL_ID,
#     messages=[
#         {
#             "role": "user",
#             "content": "What is the capital of France?"
#         }
#     ],
# )
#
# print(response.choices[0].message.content)

/home/siu856622573/.conda/envs/py10_roy/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights:   0%|          | 2/579 [00:00<01:17,  7.42it/s]/home/siu856622573/.conda/envs/py10_roy/lib/python3.10/site-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
Loading weights: 100%|██████████| 579/579 [00:02<00:00, 197.33it/s]


Model loaded.


## 2. Load Files

In [3]:
import os
import json
import re
import torch
import pandas as pd

ROI_NAMES_JSON = "../../aal_roi_names.json"                # region-name legend, matches BLEG's "{template}" field
META_CSV = "../../subject_summary.csv"

EDGE_LIST_NUMBERS = "../../edge_list_top10pos_numbers.json"   # {subject_id: "Node feature:\n...\n\nEdge feature:\n..."}
EDGE_LIST_NAMES = "../../edge_list_top10pos_names.json"  # {subject_id: "Node feature:\n...\n\nEdge feature:\n..."}
EDGE_LIST_NAMES_ONLY = "../../edge_only_from_top10pos_names.json"

ZSCORED_NUMBERS = "../../zscore_top10pos_numbers.json"
ZSCORED_NAMES = "../../zscore_top10pos_names.json"

DATASET_NAME = "ADHD-200"
LABEL1, LABEL2 = "Control", "ADHD"
HUB_GROUND_TRUTH = "../../hub_ground_truth_20roi.json"  # {subject_id: [{"roi": name, "degree": int, "cohort_z": float}, ...]}
COMMUNITY_GROUND_TRUTH = "../../community_ground_truth.json"
COMMUNITY_GROUND_TRUTH_COUNTS = "../../community_ground_truth_counts.json"
ROI_NETWORK_JSON = "../../roi_network.json"

PREPROCESS_TEMPLATE = "AAL116"
TASK_DESC = "interpreting functional connectivity patterns and reasoning about the most likely diagnostic group"
N_SUBJECTS = 768

In [4]:
# with open(EDGE_LIST_NUMBERS) as f:
# with open(EDGE_LIST_NAMES) as f:
with open(EDGE_LIST_NAMES_ONLY) as f:
# with open(COMMUNITY_GROUND_TRUTH_COUNTS) as f:
# with open(COMMUNITY_GROUND_TRUTH) as f:
# with open(HUB_GROUND_TRUTH) as f:
# with open(ZSCORED_NUMBERS) as f:
# with open(ZSCORED_NAMES) as f:
    structure_prompts_all = json.load(f)

with open("../../network_names.json") as f:
    NETWORK_NAMES = json.load(f)

if os.path.exists(ROI_NAMES_JSON):
    with open(ROI_NAMES_JSON) as f:
        roi_names = json.load(f)
    print(f"Loaded {len(roi_names)} AAL116 ROI names")
else:
    roi_names = None
    print("[warn] aal_roi_names.json not found -- description will omit the ROI-name legend")

if os.path.exists(ROI_NETWORK_JSON):
    with open(ROI_NETWORK_JSON) as f:
        roi_network = json.load(f)
    roi_to_network = dict(zip(roi_names, roi_network))
    print(f"Loaded roi_to_network for {len(roi_to_network)} ROIs")
else:
    roi_to_network = None
    print("[warn] roi_network.json not found -- community prompt will omit network labels")
# 1. Length/order sanity check
assert len(roi_names) == len(roi_network), (
    f"Length mismatch: roi_names={len(roi_names)}, roi_network={len(roi_network)} -- "
    "these two files were likely generated in different runs and are NOT safe to zip()."
)

# 2. Coverage check against the REAL edge-list data -- catches any region name
# that .get(a, "?") would otherwise silently swallow with no error.
all_edge_regions = set()
for text in structure_prompts_all.values():
    for line in text.strip().split("\n"):
        parts = line.split(":")
        if len(parts) == 3:
            all_edge_regions.add(parts[0])
            all_edge_regions.add(parts[2])

missing = all_edge_regions - set(roi_to_network.keys())
if missing:
    print(f"\n⚠️ {len(missing)} region names appear in your edge data but have NO match "
          f"in roi_to_network -- these will silently show as '?' in every prompt:")
    print(sorted(missing))
else:
    print(f"\n✅ All {len(all_edge_regions)} region names in the edge data have a valid network mapping.")


if os.path.exists(META_CSV):
    meta = pd.read_csv(META_CSV, dtype={"subject_id": str})
else:
    meta = None

subject_ids = list(structure_prompts_all.keys())[:N_SUBJECTS]
print(f"Loaded {len(structure_prompts_all)} subjects, running on first {len(subject_ids)}")
brain_graph_texts = dict(structure_prompts_all)
print("\nExample BrainGraph text (subject 1):")
print(brain_graph_texts[subject_ids[5]])

Loaded 116 AAL116 ROI names
Loaded roi_to_network for 116 ROIs

✅ All 116 region names in the edge data have a valid network mapping.
Loaded 768 subjects, running on first 768

Example BrainGraph text (subject 1):
Rectus_L:0.92:Rectus_R
Lingual_L:0.91:Lingual_R
Frontal_Sup_Orb_R:0.90:Rectus_R
Parietal_Sup_L:0.88:Parietal_Sup_R
Paracentral_Lobule_L:0.88:Paracentral_Lobule_R
Frontal_Med_Orb_R:0.87:Rectus_R
Cerebelum_Crus2_L:0.87:Cerebelum_Crus2_R
Cingulum_Ant_L:0.87:Cingulum_Ant_R
Cerebelum_4_5_L:0.86:Cerebelum_4_5_R
Frontal_Med_Orb_L:0.86:Frontal_Med_Orb_R
Cingulum_Mid_L:0.85:Cingulum_Mid_R
Cingulum_Post_L:0.83:Cingulum_Post_R
Frontal_Sup_Medial_L:0.83:Frontal_Sup_Medial_R
Cerebelum_8_L:0.83:Cerebelum_8_R
Insula_R:0.83:Putamen_R
Cerebelum_9_L:0.82:Cerebelum_9_R
Olfactory_L:0.81:Olfactory_R
Frontal_Sup_Orb_R:0.81:Rectus_L
Cerebelum_7b_L:0.81:Cerebelum_8_L
Cerebelum_3_L:0.81:Vermis_1_2
ParaHippocampal_L:0.81:ParaHippocampal_R
Cuneus_R:0.80:Occipital_Sup_L
Supp_Motor_Area_L:0.80:Supp_Motor

## 3. Prompt Templates -- Hub Identification (GraphArena-style, 1-shot CoT) + Classification (independent prompt, no hub info passed in)

In [5]:
TOP_K_HUBS = 20

In [6]:
# -------------------------------------------------------------------
# 3.1 Hub-Identification Prompt
#
# Structure follows GraphArena (Tang et al., ICLR 2025):
#   [task definition] -> [worked Example] -> [Problem to Solve] -> [format]
#
# The worked example uses a small toy graph to demonstrate the counting
# procedure -- it does not reflect the scale of the real problem. The
# full 116-region list and real edge list appear only in "Problem to Solve".
# -------------------------------------------------------------------

HUB_EXAMPLE = """**Example**
- Regions in the network: Frontal_Sup_L, Frontal_Sup_R, Parietal_Inf_L, Occipital_Mid_L, Temporal_Sup_R
- Functional connections: Frontal_Sup_L to Parietal_Inf_L, Frontal_Sup_L to Occipital_Mid_L, Frontal_Sup_L to Temporal_Sup_R, Parietal_Inf_L to Occipital_Mid_L

In this network, we have the following regions and connections: Frontal_Sup_L, Frontal_Sup_R, Parietal_Inf_L, Occipital_Mid_L, and Temporal_Sup_R. The connections are: Frontal_Sup_L to Parietal_Inf_L, Frontal_Sup_L to Occipital_Mid_L, Frontal_Sup_L to Temporal_Sup_R, and Parietal_Inf_L to Occipital_Mid_L. Counting connections per region: Frontal_Sup_L appears 3 times, Parietal_Inf_L appears 2 times, Occipital_Mid_L appears 2 times, Temporal_Sup_R appears 1 time, and Frontal_Sup_R appears 0 times. The region with the highest connection count is Frontal_Sup_L. Therefore, the top hub is [Frontal_Sup_L].
"""


def reformat_edges_to_graph_arena_style(raw_edge_text):
    """
    Converts 'RegionA:weight:RegionB' lines into GraphArena's tested
    'RegionA to RegionB' phrasing. Weight is dropped for this task since
    hub rank here is defined by degree (connection count), not strength.
    """
    lines = [ln.strip() for ln in raw_edge_text.strip().split("\n") if ln.strip()]
    pairs = []
    for line in lines:
        parts = line.split(":")
        if len(parts) == 3:
            a, _w, b = parts
            pairs.append(f"{a} to {b}")
        else:
            pairs.append(line)
    return ", ".join(pairs)


def build_hub_prompt(brain_graph_text, roi_names, top_k=TOP_K_HUBS):
    region_list_str = ", ".join(roi_names) if roi_names else "(region legend unavailable)"
    edges_str = reformat_edges_to_graph_arena_style(brain_graph_text)

    return f"""You are required to identify hub regions in the given brain functional connectivity network and output the top hub regions ranked by connectivity.

A hub region is a node with a high number of connections to other regions in the network. To identify hubs, count how many times each region name appears across the connection list -- regions appearing most frequently are the most connected.

{HUB_EXAMPLE}
**Problem to Solve**
- Regions in the network: {region_list_str}
- Functional connections: {edges_str}

Identify the top {top_k} hub regions in this network, ranked from most to least connected. Present your answer in the following format: [Region1, Region2, Region3, ..., Region{top_k}]"""


# -------------------------------------------------------------------
# 3.2 Classification Prompt (independent of hub-finding)
#
# The model sees only the raw edge list -- no hub information is passed
# in -- so the two tasks can be scored independently without one
# contaminating the other.
# -------------------------------------------------------------------


def build_classification_prompt(brain_graph_text, hub_regions):
    hub_line = ""
    if hub_regions:
        hub_str = ", ".join(hub_regions)
        hub_line = f"\nIdentified hub regions (highest connectivity): {hub_str}\n"

    return f"""Template: This data comes from the {DATASET_NAME} dataset, preprocessed using the {PREPROCESS_TEMPLATE} brain atlas template (116 regions of interest).
Description: Functional connectivity edges for one subject from resting-state fMRI. Format: RegionA:weight:RegionB, identified by their anatomical region name (AAL atlas).
Task: Based on the edge list below{" and the identified hub regions" if hub_regions else ""}, predict Control or ADHD with a confidence score for each class.
Request: Output only this JSON:
{{
"prediction": "{LABEL1} or {LABEL2}",
"class_confidence": {{
"{LABEL1}": 0.0,
"{LABEL2}": 0.0
}},
"reasoning": "(1-2 sentence)"
}}
Edges:
{brain_graph_text}
{hub_line}"""

In [7]:
## 4. Response parsers

# ==================== DeepSeek-R1-Distill-Qwen-14B (ACTIVE) ====================
def parse_json(raw_text):
    # grab the reasoning trace, if the model produced one
    think_match = re.search(r"<think>(.*?)</think>", raw_text, re.DOTALL)
    reasoning = think_match.group(1).strip() if think_match else None

    match = re.search(r"\{.*\}", raw_text, re.DOTALL)
    if not match:
        return None
    try:
        parsed = json.loads(match.group(0))
    except json.JSONDecodeError:
        return None

    if not isinstance(parsed, dict):
        return None

    if reasoning:
        parsed["reasoning"] = reasoning

    return parsed


def parse_hub_list(raw, top_k=None):
    raw = raw.strip()
    match = re.search(r"\[(.*)", raw, re.DOTALL)   # only needs opening bracket now
    if not match:
        return []
    content = match.group(1)
    content = content.split("]")[0]   # trim at closing bracket if it exists
    items = [item.strip().strip('"').strip("'") for item in content.split(",")]
    items = [item for item in items if item]
    return items[:top_k] if top_k else items


# ==================== Qwen3-14B ====================
# def parse_json(raw_text):
#     # grab the reasoning trace, if the model produced one
#     think_match = re.search(r"<think>(.*?)</think>", raw_text, re.DOTALL)
#     reasoning = think_match.group(1).strip() if think_match else None
#
#     match = re.search(r"\{.*\}", raw_text, re.DOTALL)
#     if not match:
#         return None
#     try:
#         parsed = json.loads(match.group(0))
#     except json.JSONDecodeError:
#         return None
#
#     if not isinstance(parsed, dict):
#         return None
#
#     if reasoning:
#         parsed["reasoning"] = reasoning
#
#     return parsed
#
#
# def parse_hub_list(raw, top_k=None):
#     raw = raw.strip()
#     match = re.search(r"\[(.*)", raw, re.DOTALL)   # only needs opening bracket now
#     if not match:
#         return []
#     content = match.group(1)
#     content = content.split("]")[0]   # trim at closing bracket if it exists
#     items = [item.strip().strip('"').strip("'") for item in content.split(",")]
#     items = [item for item in items if item]
#     return items[:top_k] if top_k else items


# ==================== Qwen2.5-14B-Instruct ====================
# def parse_hub_list(raw):
#     """
#     Extracts a bracketed list [Region1, Region2, ...] from the model's
#     free-text response. Returns [] if no valid bracket list is found
#     (treated as 'missing' downstream).
#     """
#     raw = raw.strip()
#     match = re.search(r"\[(.*?)\]", raw, re.DOTALL)
#     if not match:
#         return []
#     items = [item.strip().strip('"').strip("'") for item in match.group(1).split(",")]
#     return [item for item in items if item]
#
#
# def parse_json(raw):
#     raw = raw.strip()
#     start = raw.find("{")
#     if start == -1:
#         return None
#     raw = raw[start:]
#
#     diff = raw.count("{") - raw.count("}")
#     if diff > 0:
#         raw += "}" * diff
#
#     try:
#         raw = re.sub(r',\s*([}\]])', r'\1', raw)
#         return json.loads(raw)
#     except json.JSONDecodeError as e:
#         print(e)
#         print(raw)
#         return None


# ==================== GPT-Luna (API) ====================
# def parse_json(raw_text):
#     # grab the reasoning trace, if the model produced one
#     think_match = re.search(r"<think>(.*?)</think>", raw_text, re.DOTALL)
#     reasoning = think_match.group(1).strip() if think_match else None
#
#     match = re.search(r"\{.*\}", raw_text, re.DOTALL)
#     if not match:
#         return None
#     try:
#         parsed = json.loads(match.group(0))
#     except json.JSONDecodeError:
#         return None
#
#     if not isinstance(parsed, dict):
#         return None
#
#     if reasoning:
#         parsed["reasoning"] = reasoning
#
#     return parsed
#
#
# def parse_hub_list(raw, top_k=None):
#     raw = raw.strip()
#     match = re.search(r"\[(.*)", raw, re.DOTALL)   # only needs opening bracket now
#     if not match:
#         return []
#     content = match.group(1)
#     content = content.split("]")[0]   # trim at closing bracket if it exists
#     items = [item.strip().strip('"').strip("'") for item in content.split(",")]
#     items = [item for item in items if item]
#     return items[:top_k] if top_k else items

In [8]:
## 5. Generation functions

# ==================== DeepSeek-R1-Distill-Qwen-14B (ACTIVE) ====================
THINK_BUDGET = 200
ANSWER_BUDGET = 300

def generate_response_safe(full_prompt, think_budget=THINK_BUDGET, answer_budget=ANSWER_BUDGET):
    messages = [{"role": "user", "content": full_prompt}]
    prompt_text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True,
    )
    inputs = tokenizer(prompt_text, return_tensors="pt").to(model.device)

    # --- Stage 1: generate up to think_budget tokens ---
    with torch.no_grad():
        think_ids = model.generate(
            **inputs,
            max_new_tokens=think_budget,
            do_sample=False, temperature=0.6, top_p=0.95,
            pad_token_id=tokenizer.eos_token_id,
        )
    think_text = tokenizer.decode(think_ids[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

    # --- If the model didn't naturally close </think>, force it closed ---
    if "</think>" not in think_text:
        think_text = think_text + "\n</think>\n"

    # --- Stage 2: continue generating, fresh budget, just for the answer ---
    continued_prompt = prompt_text + think_text
    inputs2 = tokenizer(continued_prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        answer_ids = model.generate(
            **inputs2,
            max_new_tokens=answer_budget,
            do_sample=False, temperature=0.6, top_p=0.95,
            pad_token_id=tokenizer.eos_token_id,
        )
    answer_text = tokenizer.decode(answer_ids[0][inputs2["input_ids"].shape[1]:], skip_special_tokens=True)

    del inputs, think_ids, inputs2, answer_ids
    torch.cuda.empty_cache()

    return think_text + answer_text


def generate_hub_response(brain_graph_text):
    prompt = build_hub_prompt(brain_graph_text, roi_names, top_k=TOP_K_HUBS)
    system_message = (
        "You are a neuroscience expert. Output only a bracketed list of region names in the exact format requested. No markdown, no extra commentary, no explanation outside the bracket list."
    )
    full_prompt = f"{system_message}\n\n{prompt}"
    return generate_response_safe(full_prompt, THINK_BUDGET, ANSWER_BUDGET)


def generate_classification_response(brain_graph_text, hub_regions=None):
    prompt = build_classification_prompt(brain_graph_text, hub_regions=hub_regions)
    system_message = (
        "You are a neuroscience expert. Output only one valid JSON object matching the required schema. No markdown, comments, or extra text. Must parse with json.loads()."
    )
    full_prompt = f"{system_message}\n\n{prompt}"
    return generate_response_safe(full_prompt, THINK_BUDGET, ANSWER_BUDGET)


# ==================== Qwen3-14B ====================
# THINK_BUDGET = 200
# ANSWER_BUDGET = 300
#
# def generate_response_safe(full_prompt, think_budget=THINK_BUDGET, answer_budget=ANSWER_BUDGET):
#     messages = [{"role": "user", "content": full_prompt}]
#     prompt_text = tokenizer.apply_chat_template(
#         messages, tokenize=False, add_generation_prompt=True,
#     )
#     inputs = tokenizer(prompt_text, return_tensors="pt").to(model.device)
#
#     # --- Stage 1: generate up to think_budget tokens ---
#     with torch.no_grad():
#         think_ids = model.generate(
#             **inputs,
#             max_new_tokens=think_budget,
#             do_sample=False, temperature=0.6, top_p=0.95,
#             pad_token_id=tokenizer.eos_token_id,
#         )
#     think_text = tokenizer.decode(think_ids[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
#
#     # --- If the model didn't naturally close </think>, force it closed ---
#     if "</think>" not in think_text:
#         think_text = think_text + "\n</think>\n"
#
#     # --- Stage 2: continue generating, fresh budget, just for the answer ---
#     continued_prompt = prompt_text + think_text
#     inputs2 = tokenizer(continued_prompt, return_tensors="pt").to(model.device)
#     with torch.no_grad():
#         answer_ids = model.generate(
#             **inputs2,
#             max_new_tokens=answer_budget,
#             do_sample=False, temperature=0.6, top_p=0.95,
#             pad_token_id=tokenizer.eos_token_id,
#         )
#     answer_text = tokenizer.decode(answer_ids[0][inputs2["input_ids"].shape[1]:], skip_special_tokens=True)
#
#     del inputs, think_ids, inputs2, answer_ids
#     torch.cuda.empty_cache()
#
#     return think_text + answer_text
#
#
# def generate_hub_response(brain_graph_text):
#     prompt = build_hub_prompt(brain_graph_text, roi_names, top_k=TOP_K_HUBS)
#     system_message = (
#         "You are a neuroscience expert. Output only a bracketed list of region names in the exact format requested. No markdown, no extra commentary, no explanation outside the bracket list."
#     )
#     full_prompt = f"{system_message}\n\n{prompt}"
#     return generate_response_safe(full_prompt, THINK_BUDGET, ANSWER_BUDGET)
#
#
# def generate_classification_response(brain_graph_text, hub_regions=None):
#     prompt = build_classification_prompt(brain_graph_text, hub_regions=hub_regions)
#     system_message = (
#         "You are a neuroscience expert. Output only one valid JSON object matching the required schema. No markdown, comments, or extra text. Must parse with json.loads()."
#     )
#     full_prompt = f"{system_message}\n\n{prompt}"
#     return generate_response_safe(full_prompt, THINK_BUDGET, ANSWER_BUDGET)


# ==================== Qwen2.5-14B-Instruct ====================
# MAX_NEW_TOKENS_HUB = 400
# MAX_NEW_TOKENS_CLS = 400
#
# def generate_response_safe(prompt, system_message, max_new_tokens):
#     messages = [
#         {"role": "system", "content": system_message},
#         {"role": "user", "content": prompt},
#     ]
#     prompt_text = tokenizer.apply_chat_template(
#         messages, tokenize=False, add_generation_prompt=True,
#     )
#     inputs = tokenizer(prompt_text, return_tensors="pt").to(model.device)
#     with torch.no_grad():
#         output_ids = model.generate(
#             **inputs,
#             max_new_tokens=max_new_tokens,
#             do_sample=False,
#             pad_token_id=tokenizer.eos_token_id,
#         )
#     generated = output_ids[0][inputs["input_ids"].shape[1]:]
#     text = tokenizer.decode(generated, skip_special_tokens=True)
#
#     del inputs, output_ids, generated
#     torch.cuda.empty_cache()
#
#     return text
#
#
# def generate_hub_response(brain_graph_text):
#     prompt = build_hub_prompt(brain_graph_text, roi_names, top_k=TOP_K_HUBS)
#     system_message = (
#         "You are a neuroscience expert. Output only a bracketed list of region names in the exact format requested. No markdown, no extra commentary, no explanation outside the bracket list."
#     )
#     return generate_response_safe(prompt, system_message, MAX_NEW_TOKENS_HUB)
#
#
# def generate_classification_response(brain_graph_text, hub_regions=None):
#     prompt = build_classification_prompt(brain_graph_text, hub_regions=hub_regions)
#     system_message = (
#         "You are a neuroscience expert. Output only one valid JSON object matching the required schema. No markdown, comments, or extra text. Must parse with json.loads()."
#     )
#     return generate_response_safe(prompt, system_message, MAX_NEW_TOKENS_CLS)


# ==================== GPT-Luna (API) ====================
# import time
#
# RETRY_ATTEMPTS = 2
# MAX_NEW_TOKENS_HUB = 1500   # hub-identification response budget
# MAX_NEW_TOKENS_CLS = 500
#
# def generate_response_safe(prompt, system_message, max_new_tokens, retries=RETRY_ATTEMPTS):
#     full_prompt = f"{system_message}\n\n{prompt}"
#     for attempt in range(retries + 1):
#         try:
#             response = client.chat.completions.create(
#                 model=MODEL_ID,
#                 messages=[{"role": "user", "content": full_prompt}],
#                 max_completion_tokens=max_new_tokens,
#                 reasoning_effort="low",
#             )
#             return response.choices[0].message.content
#         except Exception:
#             if attempt == retries:
#                 raise
#             time.sleep(2 * (attempt + 1))
#
#
# def generate_hub_response(brain_graph_text):
#     prompt = build_hub_prompt(brain_graph_text, roi_names, top_k=TOP_K_HUBS)
#     system_message = (
#         "You are a neuroscience expert. Output only a bracketed list of region names in the exact format requested. No markdown, no extra commentary, no explanation outside the bracket list."
#     )
#     return generate_response_safe(prompt, system_message, MAX_NEW_TOKENS_HUB)
#
#
# def generate_classification_response(brain_graph_text, hub_regions=None):
#     prompt = build_classification_prompt(brain_graph_text, hub_regions=hub_regions)
#     system_message = (
#         "You are a neuroscience expert. Output only one valid JSON object matching the required schema. No markdown, comments, or extra text. Must parse with json.loads()."
#     )
#     return generate_response_safe(prompt, system_message, MAX_NEW_TOKENS_CLS)

In [9]:
## 6. SANITY CHECK -- run both prompts on ONE subject, inspect input + output
#     Do this before launching the full 768-subject loop.

def _count_tokens(text):
    # Works for the local HF models (uses `tokenizer`); falls back to a
    # tiktoken estimate for the API model, which has no `tokenizer` object.
    try:
        return str(len(tokenizer(text)["input_ids"]))
    except NameError:
        try:
            import tiktoken
            enc = tiktoken.get_encoding("o200k_base")
            return f"~{len(enc.encode(text))} (estimated)"
        except Exception:
            return "n/a"


sanity_sid = subject_ids[0]
sanity_edge_text = brain_graph_texts[sanity_sid]

print("=" * 90)
print(f"SANITY CHECK -- Subject {sanity_sid}")
print("=" * 90)

# ---- Step 1: Hub prompt -- input ----
hub_prompt_preview = build_hub_prompt(sanity_edge_text, roi_names, top_k=TOP_K_HUBS)
print("\n--- [STEP 1] HUB PROMPT (INPUT) ---")
print(hub_prompt_preview)
print(f"\n[hub prompt tokens: {_count_tokens(hub_prompt_preview)}]")

# ---- Step 1: Hub prompt -- output ----
hub_raw_sanity = generate_hub_response(sanity_edge_text)
hub_parsed_sanity = parse_hub_list(hub_raw_sanity)

print("\n--- [STEP 1] HUB RESPONSE (RAW LLM OUTPUT) ---")
print(hub_raw_sanity)
print("\n--- [STEP 1] HUB RESPONSE (PARSED) ---")
print(hub_parsed_sanity)

# ---- Step 2: Classification prompt -- input (chained with hub output) ----
cls_prompt_preview = build_classification_prompt(sanity_edge_text, hub_regions=hub_parsed_sanity)
print("\n" + "=" * 90)
print("--- [STEP 2] CLASSIFICATION PROMPT (INPUT, CHAINED WITH HUB OUTPUT) ---")
print(cls_prompt_preview)
print(f"\n[classification prompt tokens: {_count_tokens(cls_prompt_preview)}]")

# ---- Step 2: Classification prompt -- output ----
cls_raw_sanity = generate_classification_response(sanity_edge_text, hub_regions=hub_parsed_sanity)
cls_parsed_sanity = parse_json(cls_raw_sanity)

print("\n--- [STEP 2] CLASSIFICATION RESPONSE (RAW LLM OUTPUT) ---")
print(cls_raw_sanity)
print("\n--- [STEP 2] CLASSIFICATION RESPONSE (PARSED) ---")
print(cls_parsed_sanity)

print("\n" + "=" * 90)
print("SANITY CHECK COMPLETE -- review both steps above before running full loop.")
print("=" * 90)

[transformers] The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


SANITY CHECK -- Subject 1018959

--- [STEP 1] HUB PROMPT (INPUT) ---
You are required to identify hub regions in the given brain functional connectivity network and output the top hub regions ranked by connectivity.

A hub region is a node with a high number of connections to other regions in the network. To identify hubs, count how many times each region name appears across the connection list -- regions appearing most frequently are the most connected.

**Example**
- Regions in the network: Frontal_Sup_L, Frontal_Sup_R, Parietal_Inf_L, Occipital_Mid_L, Temporal_Sup_R
- Functional connections: Frontal_Sup_L to Parietal_Inf_L, Frontal_Sup_L to Occipital_Mid_L, Frontal_Sup_L to Temporal_Sup_R, Parietal_Inf_L to Occipital_Mid_L

In this network, we have the following regions and connections: Frontal_Sup_L, Frontal_Sup_R, Parietal_Inf_L, Occipital_Mid_L, and Temporal_Sup_R. The connections are: Frontal_Sup_L to Parietal_Inf_L, Frontal_Sup_L to Occipital_Mid_L, Frontal_Sup_L to Temporal_Su

/home/siu856622573/.conda/envs/py10_roy/lib/python3.10/site-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)



--- [STEP 1] HUB RESPONSE (RAW LLM OUTPUT) ---
Okay, so I need to figure out the top 20 hub regions in the given brain functional connectivity network. A hub region is one that has the most connections to other regions. The user provided a list of regions and a long list of functional connections. My task is to count how many times each region appears in these connections and then rank them from most to least connected.

First, I'll start by listing all the regions. There are a lot of them, so I'll make sure I don't miss any. Each region is either labeled with _L or _R, indicating left or right hemisphere, except for some like Olfactory_L and Olfactory_R, which are separate.

Next, I'll go through each connection and count the occurrences of each region. Each connection is a pair, like "A to B". I need to count both A and B each time they appear in a connection. For example, if "A to B" is listed, both A and
</think>
[Frontal_Sup_L, Frontal_Sup_R, Cingulum_Ant_L, Cingulum_Ant_R, Cingu

In [10]:
import os

OUTPUT_DIR = f"{MODEL_NAME}/hub"
os.makedirs(OUTPUT_DIR, exist_ok=True)

CHECKPOINT_PATH = os.path.join(OUTPUT_DIR, f"{MODEL_NAME}_hub_for_classification_checkpoint_20hub.json")

FINAL_PATH = os.path.join(OUTPUT_DIR, f"{MODEL_NAME}_hub_for_classification_20hub.json")

LOG_PATH = os.path.join(OUTPUT_DIR, f"{MODEL_NAME}_hub_for_classification_errors_20hub.log")

SAVE_EVERY = 10

In [11]:
## 7. Checkpoint / logging utilities
import time

def load_checkpoint():
    if os.path.exists(CHECKPOINT_PATH):
        with open(CHECKPOINT_PATH, "r") as f:
            loaded = json.load(f)
        print(f"Resuming from checkpoint: {len(loaded)} subjects already saved.")
        return loaded
    return {}


def save_checkpoint(results, path=CHECKPOINT_PATH):
    tmp_path = path + ".tmp"
    with open(tmp_path, "w") as f:
        json.dump(results, f, indent=2)
    os.replace(tmp_path, path)


def log_line(msg):
    ts = time.strftime("%Y-%m-%d %H:%M:%S")
    with open(LOG_PATH, "a") as f:
        f.write(f"[{ts}] {msg}\n")

In [ ]:
## 8. Inference loop -- runs both prompts independently per subject
from tqdm.auto import tqdm
import time

# --- Guard: don't restart a run that's already finished ---
if os.path.exists(FINAL_PATH):
    with open(FINAL_PATH) as f:
        results = json.load(f)
    print(f"Run already complete: {len(results)}/{len(subject_ids)} subjects in {FINAL_PATH}. Nothing to do.")
else:
    results = load_checkpoint()
    remaining_ids = [sid for sid in subject_ids if sid not in results]
    print(f"Total subjects: {len(subject_ids)} | already done: {len(results)} | remaining: {len(remaining_ids)}")

    n_success, n_hub_parse_fail, n_cls_parse_fail, n_error = 0, 0, 0, 0
    start_time = time.time()

    with open(LOG_PATH, "a") as log_f:
        pbar = tqdm(remaining_ids, desc="Subjects", unit="subj")
        for idx, sid in enumerate(pbar):
            try:
                brain_graph_text = brain_graph_texts[sid]

                if meta is not None:
                    gt = int(meta.loc[meta["subject_id"] == sid, "DX_binary"].values[0])
                    ground_truth = "Control" if gt == 0 else "ADHD"
                else:
                    ground_truth = None

                # -- step 1: hub-identification call --
                hub_raw = generate_hub_response(brain_graph_text)
                hub_parsed = parse_hub_list(hub_raw)
                hub_failed = not hub_parsed
                if hub_failed:
                    n_hub_parse_fail += 1
                    log_f.write(f"[hub_parse_fail] {sid}: {hub_raw[:300]!r}\n")

                # -- step 2: classification call, CHAINED with hub output --
                cls_raw = generate_classification_response(brain_graph_text, hub_regions=hub_parsed)
                cls_parsed = parse_json(cls_raw)
                if not cls_parsed:
                    n_cls_parse_fail += 1
                    log_f.write(f"[cls_parse_fail] {sid}: {cls_raw[:300]!r}\n")

                results[sid] = {
                    "subject_id": sid,
                    "ground_truth": ground_truth,

                    # hub-finding: parsed list if successful, else raw text
                    "hub_regions": hub_parsed if not hub_failed else None,
                    "hub_raw_output": hub_raw if hub_failed else None,

                    # classification: parsed fields if successful, else raw text
                    "prediction": cls_parsed.get("prediction") if cls_parsed else None,
                    "class_confidence": cls_parsed.get("class_confidence") if cls_parsed else None,
                    "reasoning": cls_parsed.get("reasoning") if cls_parsed else None,
                    "cls_raw_output": cls_raw if not cls_parsed else None,
                }
                n_success += 1
                log_f.flush()

            except Exception as e:
                n_error += 1
                log_f.write(f"[error] {sid}: {repr(e)}\n")
                log_f.flush()
                continue

            if (idx + 1) % SAVE_EVERY == 0:
                save_checkpoint(results)

    save_checkpoint(results)
    elapsed = time.time() - start_time
    print(f"\nDone in {elapsed/60:.1f} min | success: {n_success} | "
          f"hub parse fail: {n_hub_parse_fail} | cls parse fail: {n_cls_parse_fail} | errors: {n_error}")

    if len(results) == len(subject_ids):
        os.replace(CHECKPOINT_PATH, FINAL_PATH)
        print(f"All subjects done -> {FINAL_PATH}")
    else:
        print(f"Not all subjects done yet -- re-run this cell to resume from {CHECKPOINT_PATH}")

In [ ]:
## 9. Scoring -- classification accuracy + GraphArena-style hub scoring
VALID_LABELS = {"ADHD", "Control"}


def print_classification_stats(data):
    n = len(data)
    n_no_pred = sum(1 for r in data.values() if r.get("prediction") is None)
    malformed = [
        (sid, r.get("prediction")) for sid, r in data.items()
        if r.get("prediction") is not None and r.get("prediction") not in VALID_LABELS
    ]
    scorable = [
        r for r in data.values()
        if r.get("ground_truth") in VALID_LABELS and r.get("prediction") in VALID_LABELS
    ]
    n_correct = sum(1 for r in scorable if r["prediction"] == r["ground_truth"])

    n_hub_fallback = sum(1 for r in data.values() if r.get("hub_regions") is None)

    print("=== Classification ===")
    print(f"Total subjects:              {n}")
    print(f"No prediction (parse fail):  {n_no_pred}")
    print(f"Malformed prediction:        {len(malformed)}")
    print(f"Scorable subjects:           {len(scorable)}")
    if scorable:
        print(f"Accuracy:                    {n_correct / len(scorable):.3f}")
    print(f"Subjects w/ hub fallback (no hub info reached classification): {n_hub_fallback}")


def score_hub_response(predicted_hubs, ground_truth_top20, all_valid_regions, edge_regions):
    """
    GraphArena-style 4-bucket classification for one subject's hub response:
      missing        -- empty / unparseable response
      hallucinatory  -- contains a region not in AAL-116 legend, or not
                        present in this subject's actual edge list
      suboptimal     -- valid regions, low overlap with ground-truth top-20
      correct        -- valid regions, strong overlap with ground-truth top-20
    """
    if not predicted_hubs:
        return "missing"

    for region in predicted_hubs:
        if all_valid_regions and region not in all_valid_regions:
            return "hallucinatory"
        if edge_regions and region not in edge_regions:
            return "hallucinatory"

    overlap = len(set(predicted_hubs) & set(ground_truth_top20))
    overlap_ratio = overlap / max(len(predicted_hubs), 1)

    if overlap_ratio >= 0.7:
        return "correct (>= 0.7)"
    elif overlap_ratio >= 0.6:
        return "correct (>= 0.6)"
    elif overlap_ratio >= 0.5:
        return "correct (>= 0.5)"
    elif overlap_ratio >= 0.4:
        return "correct (>= 0.4)"
    elif overlap_ratio >= 0.3:
        return "correct (>= 0.3)"
    elif overlap_ratio >= 0.2:
        return "correct (>= 0.2)"
    else:
        return "suboptimal"


def print_hub_stats(data, ground_truth_hub_map, all_valid_regions, brain_graph_texts):
    buckets = {
        "correct (>= 0.7)": 0,
        "correct (>= 0.6)": 0,
        "correct (>= 0.5)": 0,
        "correct (>= 0.4)": 0,
        "correct (>= 0.3)": 0,
        "correct (>= 0.2)": 0,
        "suboptimal": 0,
        "hallucinatory": 0,
        "missing": 0
    }

    for sid, r in data.items():
        predicted = r.get("hub_regions", [])
        gt_top20 = ground_truth_hub_map.get(sid, [])
        edge_text = brain_graph_texts.get(sid, "")
        edge_regions = set(re.findall(r"[A-Za-z0-9_]+", edge_text))

        label = score_hub_response(predicted, gt_top20, all_valid_regions, edge_regions)
        buckets[label] += 1

    n = sum(buckets.values())
    print("\n=== Hub Identification ===")
    for label, count in buckets.items():
        pct = 100 * count / n if n else 0
        print(f"{label:14s}: {count:4d} ({pct:.1f}%)")